In [1]:
import stpsf
import warnings

import astropy.units as u
import FunctionLib as FL
import inspect
from tqdm import tqdm
import astropy
import wave
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from collections import defaultdict
import re
import scipy
from astropy.io import fits as asfits

mpl.rcParams['font.family'] = 'serif'


warnings.filterwarnings("ignore")

DJAv4Catalog = FL.Spectrum_Catalog()
DJAv4Catalog.load_from_pkl(os.path.expanduser(
    './DJAV4.2Catalog.pkl'))
print(DJAv4Catalog.sample_num())

DJAv4Catalog.to_dataframe()

**WARNING**: LOCAL JWST PRD VERSION PRDOPSSOC-068 DOESN'T MATCH THE CURRENT ONLINE VERSION PRDOPSSOC-071
Please consider updating pysiaf, e.g. pip install --upgrade pysiaf or conda update pysiaf
/home/xingyaocai/miniconda3/envs/DustCurve/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


3383


,survey_id_subid,prism_filepath,prism_redshift,determined_redshift,grating_filepaths,grating_redshifts,file_count,available_filters,properties,survey_id,grating_within_coverage,sample_flag,reason_for_exclusion,grating_slitloss_correction
0,snh0pe-v4_4446_102,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,0.2259,0.2259,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': nan, 'g235m-f170lp': nan}",3,"{prism-clear, g140m-f100lp, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,False,[redshift_below_3],{}
1,snh0pe-v4_4446_143,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.6318,1.6311,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.6313, 'g235m-f170lp': 1.6309}",3,"{prism-clear, g140m-f100lp, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,False,[redshift_below_3],{}
2,snh0pe-v4_4446_285,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,0.4446,0.4462,{'g235m-f170lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g235m-f170lp': 0.4462, 'g140m-f100lp': 0.4462}",3,"{prism-clear, g140m-f100lp, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,False,[redshift_below_3],{}
3,snh0pe-v4_4446_29,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.7834,1.77975,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.7799, 'g235m-f170lp': 1.7796}",3,"{prism-clear, g140m-f100lp, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,False,[redshift_below_3],{}
4,snh0pe-v4_4446_123,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.7855,1.7855,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.7855, 'g235m-f170lp': 1.7851}",3,"{prism-clear, g140m-f100lp, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,False,[redshift_below_3],{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43037,glimpse-obs02-v4_9223_9152,None,None,5.5385,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 5.5385},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,{},False,[no_prism_spectrum],{}
43038,glimpse-obs02-v4_9223_98001,None,None,2.6322,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 2.6322},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,{},False,[no_prism_spectrum],{}
43039,glimpse-obs02-v4_9223_47046,None,None,4.3554,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 4.3554},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,{},False,[no_prism_spectrum],{}
43040,glimpse-obs02-v4_9223_45350,None,None,1.3675,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 1.3675},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,{},False,[no_prism_spectrum],{}


In [9]:
def calculate_halpha_hbeta_psf_for_given_catalog(item):
    """
    接收 catalog 的一个条目 (id, catalog_data) 作为输入。
    返回一个包含 id 和计算结果的元组。
    """
    id, catalog = item

    if not catalog.get('sample_flag', False):
        return (id, None)

    if 'grating_within_coverage' not in catalog:
        return (id, None)

    # 这个字典将存储所有计算出的新数据
    slitloss_corrections = {}

    for filter_name_full, grating_filepath in catalog['grating_within_coverage'].items():
        filter_name = filter_name_full.split('-')[1]
        halpha_wavelength = 6564.61 * (1 + catalog['determined_redshift']) * u.AA
        hbeta_wavelength = 4862.68 * (1 + catalog['determined_redshift']) * u.AA

        try:
            with asfits.open(grating_filepath) as grating_spectrum_fits:
                x_offset_grating = grating_spectrum_fits[1].header.get('SRCXPOS', 0.0)
                y_offset_grating = grating_spectrum_fits[1].header.get('SRCYPOS', 0.0)
                slitlit_status = grating_spectrum_fits[1].header.get('SHUTSTA', '')

                nirspec = stpsf.NIRSpec()
                nirspec.options['source_offset_x'] = x_offset_grating * 0.20
                nirspec.options['source_offset_y'] = y_offset_grating * 0.46

                nirspec.filter = filter_name
                nirspec.image_mask = 'Three adjacent MSA open shutters'

                halpha_wavelength_m = halpha_wavelength.to(u.m).value
                hbeta_wavelength_m = hbeta_wavelength.to(u.m).value

                selection_mask = np.zeros((48, 48), dtype=bool)
                if slitlit_status in ['1x1', 'x11', '11x']:
                    # 假设对于这三种状态，选择区域是相同的
                    x1, x2 = 23, 25
                    y1, y2 = 22, 26
                    selection_mask[y1:y2, x1:x2] = True
                else:
                    # 如果 slitlit_status 不是预期的值，可以跳过或记录日志
                    continue

                OVERSAMPLE = 4

                halpha_psf = nirspec.calc_psf(monochromatic=halpha_wavelength_m, oversample=OVERSAMPLE)
                hbeta_psf = nirspec.calc_psf(monochromatic=hbeta_wavelength_m, oversample=OVERSAMPLE)

                halpha_selection_flux = np.nansum(halpha_psf[3].data[selection_mask])
                hbeta_selection_flux = np.nansum(hbeta_psf[3].data[selection_mask])

                halpha_total_flux = np.nansum(halpha_psf[3].data)
                hbeta_total_flux = np.nansum(hbeta_psf[3].data)

                # 避免除以零的错误
                halpha_psf_fraction = halpha_selection_flux / halpha_total_flux if halpha_total_flux != 0 else 0
                hbeta_psf_fraction = hbeta_selection_flux / hbeta_total_flux if hbeta_total_flux != 0 else 0

                slitloss_corrections[f'halpha_psf_fraction_{filter_name}'] = halpha_psf_fraction
                slitloss_corrections[f'hbeta_psf_fraction_{filter_name}'] = hbeta_psf_fraction

        except Exception as e:
            # 捕获可能的异常，例如文件读取错误，并打印出来
            print(f"Error processing id {id}, filter {filter_name}: {e}")
            continue

    # 如果有计算结果，则返回 id 和结果字典
    if slitloss_corrections:
        return (id, slitloss_corrections)
    else:
        return (id, None)


In [ ]:
for id, catalog in tqdm(DJAv4Catalog.catalog_iterator()):
    if catalog['sample_flag'] == False:
        continue

    calculate_halpha_hbeta_psf_for_given_catalog(id, catalog, DJAv4Catalog=DJAv4Catalog)

43042it [5:53:23,  2.03it/s] 


In [46]:
DJAv4Catalog.save_catalog_to_pkl(pathlib.Path('./DJAV4.2Catalog.pkl').expanduser())

In [10]:
import multiprocessing
from tqdm import tqdm

if __name__ == '__main__':  # 在 Windows 和 macOS 上使用 multiprocessing 必须把主逻辑放在这个 block 中
    # 加载你的 Catalog
    DJAv4Catalog = FL.Spectrum_Catalog()
    DJAv4Catalog.load_from_pkl(os.path.expanduser('./DJAV4.2Catalog.pkl'))
    print(f"Catalog loaded with {DJAv4Catalog.sample_num()} samples.")

    # 准备要处理的数据列表
    # 只选择 sample_flag 为 True 的条目进行计算，减少不必要的进程开销
    items_to_process = [
        (id, catalog) for id, catalog in DJAv4Catalog.catalog_iterator() if catalog.get('sample_flag', False)
    ]

    # 设置进程数，可以根据你的 CPU 核心数来定
    # os.cpu_count() 可以获取核心数
    num_processes = 64
    print(f"Starting parallel processing with {num_processes} processes...")

    # 使用进程池
    # imap 会按顺序返回结果，非常适合和 tqdm 结合使用来显示进度条
    with multiprocessing.Pool(processes=num_processes) as pool:
        # 使用 tqdm 显示进度
        results = list(tqdm(pool.imap(calculate_halpha_hbeta_psf_for_given_catalog, items_to_process), total=len(items_to_process)))

    print("All calculations finished. Updating the main catalog...")

    # 统一更新 Catalog
    update_count = 0
    for id, new_data in tqdm(results, desc="Updating catalog"):
        if new_data is not None:
            # 获取原始的 catalog 条目
            original_catalog_item = DJAv4Catalog.get_catalog_item(id)

            # 如果原始条目中没有 'grating_slitloss_correction'，则先创建一个
            if 'grating_slitloss_correction' not in original_catalog_item:
                original_catalog_item['grating_slitloss_correction'] = {}

            # 更新数据
            original_catalog_item['grating_slitloss_correction'].update(new_data)

            # 将更新后的条目写回 catalog 对象
            DJAv4Catalog.update_catalog_item(id, original_catalog_item)
            update_count += 1

    print(f"Catalog updated for {update_count} items.")

    # 在这里，如果你需要将更新后的 catalog 保存回 pkl 文件，可以调用相应的方法
    # 例如: DJAv4Catalog.save_to_pkl('DJAV4.2Catalog_updated.pkl')

    print("Process complete.")

Catalog loaded with 3383 samples.
Starting parallel processing with 64 processes...


  0%|          | 0/3383 [04:08<?, ?it/s]


KeyboardInterrupt: 

In [11]:
count=0
for id, catalog in tqdm(DJAv4Catalog.catalog_iterator()):
    if catalog['sample_flag'] == False:
        continue

    if catalog['grating_slitloss_correction'].keys():
        count+=1

print(count)

43042it [00:00, 1413105.03it/s]

31
